# 🏆 Classification Challenge: Predicting Income Level


Welcome to this **Kaggle-like machine learning challenge**!  
Your task is to build a classifier that predicts whether an individual earns more than **$50,000/year** using census data.

### Dataset
We will use the **Adult Census Income dataset** from the UCI Machine Learning Repository.  
It includes both numerical and categorical features that require **preprocessing** before modeling.

### Objective
- Perform data preprocessing (handle missing values, encode categorical features, scale numeric features).  
- Train a classification model.  
- Evaluate the performance on the test set.



In [12]:

import pandas as pd

columns = ['age','workclass','fnlwgt','education','education-num',
           'marital-status','occupation','relationship','race','sex',
           'capital-gain','capital-loss','hours-per-week',
           'native-country','salary']

train = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data',
                    names=columns, engine='python')

test = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test',
                   names=columns, engine='python', skiprows=1)

print(train.shape, test.shape)
train.head()


(32561, 15) (16281, 15)


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 🔍 Exploratory Data Analysis (EDA)

In [13]:

# Check missing values represented as '?'
print((train == '?').sum())

# Class distribution
print(train['salary'].value_counts())

# Quick statistics
train.describe(include='all')


age               0
workclass         0
fnlwgt            0
education         0
education-num     0
marital-status    0
occupation        0
relationship      0
race              0
sex               0
capital-gain      0
capital-loss      0
hours-per-week    0
native-country    0
salary            0
dtype: int64
salary
<=50K    24720
>50K      7841
Name: count, dtype: int64


,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
count,32561.000000,32561,3.256100e+04,32561,32561.000000,32561,32561,32561,32561,32561,32561.000000,32561.000000,32561.000000,32561,32561
unique,NaN,9,NaN,16,NaN,7,15,6,5,2,NaN,NaN,NaN,42,2
top,NaN,Private,NaN,HS-grad,NaN,Married-civ-spouse,Prof-specialty,Husband,White,Male,NaN,NaN,NaN,United-States,<=50K
freq,NaN,22696,NaN,10501,NaN,14976,4140,13193,27816,21790,NaN,NaN,NaN,29170,24720
mean,38.581647,NaN,1.897784e+05,NaN,10.080679,NaN,NaN,NaN,NaN,NaN,1077.648844,87.303830,40.437456,NaN,NaN
std,13.640433,NaN,1.055500e+05,NaN,2.572720,NaN,NaN,NaN,NaN,NaN,7385.292085,402.960219,12.347429,NaN,NaN
min,17.000000,NaN,1.228500e+04,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,1.000000,NaN,NaN
25%,28.000000,NaN,1.178270e+05,NaN,9.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,40.000000,NaN,NaN
50%,37.000000,NaN,1.783560e+05,NaN,10.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,40.000000,NaN,NaN
75%,48.000000,NaN,2.370510e+05,NaN,12.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,45.000000,NaN,NaN


## ⚙️ Preprocessing & Baseline Model

In [24]:

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

X_train = train.drop('salary', axis=1)
y_train = train['salary']
X_test = test.drop('salary', axis=1)
y_test = test['salary'].str.replace('.', '', regex=False)  # fix labels in test set

numeric_feats = ['age','fnlwgt','education-num','capital-gain','capital-loss','hours-per-week']
categorical_feats = list(set(columns) - set(numeric_feats) - {'salary'})

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_feats),
    ('cat', categorical_transformer, categorical_feats)
])

model = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=500))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

       <=50K       0.88      0.93      0.91     12435
        >50K       0.73      0.60      0.66      3846

    accuracy                           0.85     16281
   macro avg       0.81      0.77      0.78     16281
weighted avg       0.85      0.85      0.85     16281




## ✅ Next Steps for You
- Try different models: RandomForest, GradientBoosting, or XGBoost.  
- Handle missing values more creatively.  
- Feature engineering: Group categories, bin ages, etc.  
- Hyperparameter tuning.  

The baseline above is just a starting point – can you beat it?


Here we basically did everything the same except for encoding the targers (y) before passing it to the pipeline since xgboost cannot handle them same as logistic regression pipeline

In [34]:
%pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.4/247.4 kB 17.3 MB/s eta 0:00:00


In [39]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import numpy as np
from lightgbm import LGBMClassifier

# Prepare data
X_train = train.drop('salary', axis=1)
y_train = train['salary']
X_test = test.drop('salary', axis=1)
y_test = test['salary'].str.replace('.', '', regex=False)  # clean labels

X_train['capital-gain'] = np.log1p(X_train['capital-gain'])
X_test['capital-gain'] = np.log1p(X_test['capital-gain'])
X_train['capital-loss'] = np.log1p(X_train['capital-loss'])
X_test['capital-loss'] = np.log1p(X_test['capital-loss'])

X_train['age_bin'] = pd.cut(X_train['age'], bins=[0,25,35,45,55,65,100], labels=False)
X_test['age_bin'] = pd.cut(X_test['age'], bins=[0,25,35,45,55,65,100], labels=False)

# Encode target variable into 0/1
le = LabelEncoder()
y_train_2 = le.fit_transform(y_train)
y_test_2 = le.transform(y_test)

numeric_feats = ['fnlwgt','education-num','capital-gain','capital-loss','hours-per-week']
categorical_feats = list(set(columns) - set(numeric_feats) - {'salary'})

# Transformers
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_feats),
    ('cat', categorical_transformer, categorical_feats)
])

# Stronger model
model2 = Pipeline([
    ('prep', preprocessor),
    ('clf', LGBMClassifier(
        n_estimators=434,
        learning_rate=0.10608175245316723,
        max_depth=-4,
        subsample=0.8301813141788214,
        colsample_bytree=0.8565762123875033,
        random_state=42
        ))
    ])

# Train
model2.fit(X_train, y_train_2)

# Evaluate
y_pred = model2.predict(X_test)
print(classification_report(y_test_2, y_pred, target_names=le.classes_))

[LightGBM] [Info] Number of positive: 7841, number of negative: 24720
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009386 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 841
[LightGBM] [Info] Number of data points in the train set: 32561, number of used features: 160
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.240810 -> initscore=-1.148246
[LightGBM] [Info] Start training from score -1.148246


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


              precision    recall  f1-score   support

       <=50K       0.90      0.94      0.92     12435
        >50K       0.76      0.66      0.70      3846

    accuracy                           0.87     16281
   macro avg       0.83      0.80      0.81     16281
weighted avg       0.87      0.87      0.87     16281



# Comparison

In [41]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

def get_metrics(model, X_test, y_test, label_encoder=None, name="Model"):
    """Return classification metrics as a DataFrame row"""
    y_pred = model.predict(X_test)

    if label_encoder is not None:  # if target is encoded, decode back for consistency
        y_true = label_encoder.inverse_transform(y_test)
        y_pred = label_encoder.inverse_transform(y_pred)
    else:
        y_true = y_test

    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=list(set(y_true)))

    results = []
    for i, label in enumerate(sorted(set(y_true))):
        results.append({
            "model": name,
            "class": label,
            "precision": precision[i],
            "recall": recall[i],
            "f1-score": f1[i],
            "support": support[i]
        })

    # Add overall accuracy
    results.append({
        "model": name,
        "class": "accuracy",
        "precision": "",
        "recall": "",
        "f1-score": accuracy_score(y_true, y_pred),
        "support": len(y_true)
    })

    return pd.DataFrame(results)

# Example: compare Logistic Regression vs XGBoost
baseline_results = get_metrics(model, X_test, y_test, name="LogReg")
xgb_results = get_metrics(model2, X_test, y_test_2, label_encoder=le, name="LGBM")

# Merge results
comparison = pd.concat([baseline_results, xgb_results])
comparison_pivot = comparison.pivot(index="class", columns="model", values=["precision","recall","f1-score","support"])

comparison_pivot

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


precision              recall            f1-score           support  \
model         LGBM    LogReg      LGBM    LogReg      LGBM    LogReg    LGBM   
class                                                                          
 <=50K    0.897885  0.854491  0.935505  0.944029  0.916309  0.897031   12435   
 >50K     0.758797  0.726308  0.656006  0.480239  0.703668  0.578181    3846   
accuracy                                           0.86948   0.83447   16281   

                 
model    LogReg  
class            
 <=50K    12435  
 >50K      3846  
accuracy  16281